In [ ]:
# ==============================================================
# 10 – Business Metrics & Portfolio Impact
# Translates MARL decisions into bank-level financial metrics
# Supports RQ4 strongly + executive reporting
# Production-bank ready
# ==============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

print("Business Metrics Notebook – Production Version")

# --------------------------------------------------------------
# 1. Load Data & Model
# --------------------------------------------------------------
X = np.load(DATA_PROCESSED / "X_fused.npy").astype(np.float32)
y = np.load(DATA_PROCESSED / "y.npy")
thin = np.load(DATA_PROCESSED / "thin.npy")
df = pd.read_csv(DATA_SYNTHETIC / "global_credit_from_german.csv")

# Reload Hierarchical MARL (same architecture)
class SpecializedAgent(nn.Module):
    def __init__(self, state_dim, hidden=128, action_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, action_dim)
        )
    def forward(self, x): return self.net(x)

class AttentionCoordinator(nn.Module):
    def __init__(self, num_agents, action_dim=3):
        super().__init__()
        self.query = nn.Linear(action_dim, action_dim)
        self.key   = nn.Linear(action_dim, action_dim)
        self.value = nn.Linear(action_dim, action_dim)
        self.scale = action_dim ** 0.5
        self.out   = nn.Sequential(nn.Linear(action_dim, 64), nn.ReLU(), nn.Linear(64, action_dim))
    def forward(self, agent_logits):
        x = agent_logits.permute(1, 0, 2)
        Q, K, V = self.query(x), self.key(x), self.value(x)
        attn = torch.softmax(torch.bmm(Q, K.transpose(1,2)) / self.scale, dim=-1)
        out = torch.bmm(attn, V).mean(dim=1)
        return self.out(out), attn

class HierarchicalMARL(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super().__init__()
        self.agent_names = ["Risk", "Affordability", "Macro", "Fairness", "Pricing"]
        self.agents = nn.ModuleDict({n: SpecializedAgent(state_dim) for n in self.agent_names})
        self.coordinator = AttentionCoordinator(len(self.agent_names), action_dim)
        self.value_head = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, state):
        agent_outs = torch.stack([self.agents[n](state) for n in self.agent_names])
        logits, attn = self.coordinator(agent_outs)
        value = self.value_head(state).squeeze(-1)
        return logits, value, attn, agent_outs

state_dim = X.shape[1]
model = HierarchicalMARL(state_dim).to(device)
ckpt = torch.load(RESULTS / "hierarchical_marl_final.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("✓ Hierarchical MARL loaded")

# --------------------------------------------------------------
# 2. Generate Decisions for Full Portfolio
# --------------------------------------------------------------
def get_decisions(model, X):
    actions, probs = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), 256):
            batch = torch.tensor(X[i:i+256], device=device)
            logits, _, _, _ = model(batch)
            p = F.softmax(logits, dim=1)
            actions.append(p.argmax(1).cpu().numpy())
            probs.append(p.cpu().numpy())
    return np.concatenate(actions), np.concatenate(probs)

actions, probs = get_decisions(model, X)
print(f"Decisions generated for {len(actions):,} applicants")

# --------------------------------------------------------------
# 3. Core Business Metrics Engine
# --------------------------------------------------------------
# Assumptions (can be calibrated per country later)
AVG_LOAN = 5500
INTEREST_MARGIN = 0.135          # net interest margin
LGD = 0.60                       # Loss Given Default
OPERATING_COST_PER_LOAN = 120

df_portfolio = df.copy()
df_portfolio["action"] = actions
df_portfolio["approve"] = (actions == 1).astype(int)
df_portfolio["counter"] = (actions == 2).astype(int)
df_portfolio["reject"]  = (actions == 0).astype(int)
df_portfolio["prob_approve"] = probs[:, 1]

# Financial calculations
df_portfolio["loan_amount"] = df_portfolio.get("credit_amount", AVG_LOAN)
df_portfolio["expected_interest"] = df_portfolio["loan_amount"] * INTEREST_MARGIN
df_portfolio["expected_loss"] = np.where(
    (df_portfolio["approve"] == 1) & (df_portfolio["default"] == 1),
    df_portfolio["loan_amount"] * LGD,
    0
)

# Portfolio level
total_approved = df_portfolio["approve"].sum()
total_volume = df_portfolio.loc[df_portfolio["approve"]==1, "loan_amount"].sum()
total_interest = df_portfolio.loc[df_portfolio["approve"]==1, "expected_interest"].sum()
total_expected_loss = df_portfolio["expected_loss"].sum()
total_operating_cost = total_approved * OPERATING_COST_PER_LOAN

net_profit = total_interest - total_expected_loss - total_operating_cost
roi = net_profit / (total_volume + 1e-8)

# Thin-file specific
thin_mask = df_portfolio["thin_file"] == 1
thin_approved = df_portfolio.loc[thin_mask, "approve"].sum()
thin_total = thin_mask.sum()
thin_approval_rate = thin_approved / (thin_total + 1e-8)

print("\n=== Portfolio Summary ===")
print(f"Total Applicants       : {len(df_portfolio):,}")
print(f"Approved               : {total_approved:,} ({total_approved/len(df_portfolio):.1%})")
print(f"Thin-file Approval Rate: {thin_approval_rate:.1%}")
print(f"Total Volume           : ${total_volume:,.0f}")
print(f"Expected Interest      : ${total_interest:,.0f}")
print(f"Expected Loss          : ${total_expected_loss:,.0f}")
print(f"Net Profit Proxy       : ${net_profit:,.0f}")
print(f"ROI Proxy              : {roi:.2%}")

# --------------------------------------------------------------
# 4. Country-level Profitability
# --------------------------------------------------------------
country_metrics = []
for country in df_portfolio["country"].unique():
    sub = df_portfolio[df_portfolio["country"] == country]
    approved = sub["approve"].sum()
    volume = sub.loc[sub["approve"]==1, "loan_amount"].sum()
    interest = sub.loc[sub["approve"]==1, "expected_interest"].sum()
    exp_loss = sub["expected_loss"].sum()
    profit = interest - exp_loss - (approved * OPERATING_COST_PER_LOAN)
    thin_rate = sub.loc[sub["thin_file"]==1, "approve"].mean()

    country_metrics.append({
        "Country": country,
        "Applicants": len(sub),
        "Approval Rate": approved / len(sub),
        "Thin-file Approval": thin_rate,
        "Volume": volume,
        "Expected Loss": exp_loss,
        "Net Profit": profit,
        "ROI": profit / (volume + 1e-8)
    })

country_df = pd.DataFrame(country_metrics).sort_values("Net Profit", ascending=False)
print("\n=== Country-level Performance ===")
display(country_df.round(3))

country_df.to_csv(RESULTS / "business_metrics_by_country.csv", index=False)

# --------------------------------------------------------------
# 5. Risk-Return Trade-off Simulation
# --------------------------------------------------------------
thresholds = np.linspace(0.2, 0.8, 13)
tradeoff = []

for th in thresholds:
    # Simulate threshold on approve probability
    sim_approve = (df_portfolio["prob_approve"] >= th).astype(int)
    vol = df_portfolio.loc[sim_approve==1, "loan_amount"].sum()
    interest = df_portfolio.loc[sim_approve==1, "expected_interest"].sum()
    loss = df_portfolio.loc[(sim_approve==1) & (df_portfolio["default"]==1), "loan_amount"].sum() * LGD
    profit = interest - loss - (sim_approve.sum() * OPERATING_COST_PER_LOAN)
    approval_rate = sim_approve.mean()
    thin_app = df_portfolio.loc[(sim_approve==1) & (df_portfolio["thin_file"]==1)].shape[0] / (thin_total + 1e-8)

    tradeoff.append({
        "Threshold": th,
        "Approval Rate": approval_rate,
        "Thin-file Approval": thin_app,
        "Expected Loss": loss,
        "Net Profit": profit
    })

tradeoff_df = pd.DataFrame(tradeoff)

# --------------------------------------------------------------
# 6. Executive Visualizations
# --------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Profit by Country
sns.barplot(data=country_df, x="Country", y="Net Profit", ax=axes[0,0], palette="viridis")
axes[0,0].set_title("Net Profit by Country")
axes[0,0].tick_params(axis='x', rotation=45)

# Thin-file Approval vs Profit
axes[0,1].scatter(country_df["Thin-file Approval"], country_df["Net Profit"], s=120, c=country_df["ROI"], cmap="coolwarm")
axes[0,1].set_xlabel("Thin-file Approval Rate")
axes[0,1].set_ylabel("Net Profit")
axes[0,1].set_title("Inclusion vs Profitability (RQ4)")

# Risk-Return curve
axes[1,0].plot(tradeoff_df["Approval Rate"], tradeoff_df["Net Profit"], marker='o', lw=2)
axes[1,0].set_xlabel("Approval Rate")
axes[1,0].set_ylabel("Net Profit")
axes[1,0].set_title("Approval Rate vs Net Profit Trade-off")
axes[1,0].grid(True)

# Expected Loss vs Approval
axes[1,1].plot(tradeoff_df["Approval Rate"], tradeoff_df["Expected Loss"], marker='o', color="red", lw=2)
axes[1,1].set_xlabel("Approval Rate")
axes[1,1].set_ylabel("Expected Loss")
axes[1,1].set_title("Approval Rate vs Expected Loss")
axes[1,1].grid(True)

plt.tight_layout()
plt.savefig(RESULTS / "business_metrics_executive_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# --------------------------------------------------------------
# 7. Final Executive Summary
# --------------------------------------------------------------
summary = {
    "Total Applicants": len(df_portfolio),
    "Approval Rate": total_approved / len(df_portfolio),
    "Thin-file Approval Rate": thin_approval_rate,
    "Total Lending Volume": total_volume,
    "Expected Interest Income": total_interest,
    "Expected Credit Loss": total_expected_loss,
    "Net Profit Proxy": net_profit,
    "ROI Proxy": roi,
    "Average Loan Size": total_volume / (total_approved + 1e-8)
}

summary_df = pd.Series(summary)
print("\n=== EXECUTIVE SUMMARY ===")
print(summary_df.round(3))

summary_df.to_csv(RESULTS / "executive_business_summary.csv")
tradeoff_df.to_csv(RESULTS / "risk_return_tradeoff.csv", index=False)

print("\n✅ 10_Business_Metrics completed successfully.")
print("All executive reports saved to results/ folder.")
print("This notebook completes the full research pipeline and supports RQ4.")